#Init / Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, to_date
from pyspark.sql.types import StringType

# Reading prd_info table from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_prd_info_raw")

In [0]:
df.printSchema()

#Transforming data

## Renaming the columns

In [0]:
FIELD_RENAME = {
    "prd_id" : "Id",
    "prd_key":"Key",
    "prd_nm" :"Name",
    "prd_cost":"Cost",
    "prd_line" : "Line",
    "prd_start_dt":"Start_Date",
    "prd_end_dt":"End_Date"
}

for old_name, new_name in FIELD_RENAME.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming the data

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.display()

### Checking date format of start and end date

In [0]:
df.select("Start_Date", "End_Date").show(10)


In [0]:
df.select("Start_Date").filter(col("Start_Date").substr(6, 2).cast("int") > 12).show(5)

## Casting col to appropriate type and format

In [0]:
df = (
    df
    .withColumn("Cost", col("Cost").cast("int"))
    .withColumn("Start_Date", to_date("Start_Date", "yyyy-MM-dd"))
    .withColumn("End_Date", to_date("End_Date", "yyyy-MM-dd"))
)

In [0]:
df.display()

# Writing data to silver

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.silver.crm_prd_info")
)